In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(device)

mps


# Define the Class
- nn.Modlue: 重みの管理やGPU転送、train/evalの切り替えなど
- ここでは3層の全結合ネットワークを構築

In [3]:
class NearalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NearalNetwork().to(device)
print(model)

NearalNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
# forward
x = torch.rand(1, 28, 28, device=device)
logits = model(x)
print(logits)

print(logits.shape)  # torch.Size(1, 10)

pred_prob = nn.Softmax(dim=1)(logits)  # dim=1, すなわち10の方に適用
y_pred = pred_prob.argmax(1)  # dim=1, すなわち10の方に適用
print(f"predicted class: {y_pred}")

tensor([[-0.0076, -0.0244,  0.0399, -0.0678, -0.0038,  0.0861, -0.0591,  0.0713,
         -0.0099,  0.0872]], device='mps:0', grad_fn=<LinearBackward0>)
torch.Size([1, 10])
predicted class: tensor([9], device='mps:0')


In [6]:
input_image = torch.rand(3, 28, 28)
print(input_image.shape)

torch.Size([3, 28, 28])


In [7]:
# nn.Flatten
flatten = nn.Flatten()
flat_image = flatten(input_image)  # 3,28,28 => 3, 784
print(flat_image.shape)

torch.Size([3, 784])


In [8]:
# nn.Linear
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.shape)

torch.Size([3, 20])


In [9]:
# nn.ReLU
print(f"before ReLU: {hidden1}\n")
hidden1_relu = nn.ReLU()(hidden1)
print(f"after ReLU: {hidden1_relu}")

before ReLU: tensor([[ 0.1820, -0.2355, -0.0665,  0.2487, -0.3792,  0.3703, -0.1016,  0.2398,
          0.1283,  0.1158,  0.0191,  0.4759, -0.3669,  0.2336, -0.6721, -0.1488,
          0.5739,  0.1197,  0.1266, -0.4325],
        [ 0.2685, -0.6382, -0.0024,  0.2047, -0.1898,  0.3751, -0.0279,  0.1551,
          0.1080, -0.2264,  0.1064,  0.5545, -0.5769, -0.1300, -0.2658,  0.1618,
          0.5705,  0.3183,  0.3177, -0.3111],
        [ 0.1926, -0.9183, -0.0502, -0.0873, -0.2179,  0.3382,  0.3591,  0.5628,
          0.0592, -0.0113,  0.0747,  0.4817, -0.0177, -0.1208, -0.5100,  0.2519,
          0.5516,  0.1898,  0.4708, -0.4766]], grad_fn=<AddmmBackward0>)

after ReLU: tensor([[0.1820, 0.0000, 0.0000, 0.2487, 0.0000, 0.3703, 0.0000, 0.2398, 0.1283,
         0.1158, 0.0191, 0.4759, 0.0000, 0.2336, 0.0000, 0.0000, 0.5739, 0.1197,
         0.1266, 0.0000],
        [0.2685, 0.0000, 0.0000, 0.2047, 0.0000, 0.3751, 0.0000, 0.1551, 0.1080,
         0.0000, 0.1064, 0.5545, 0.0000, 0.0000, 0.000

In [ ]:
# nn.Sequential
# レイヤーの順番にデータが渡される。forward時、全てのレイヤを記述しなくて済む
seq_modules = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 20),
    nn.ReLU(),
    nn.Linear(20, 10)
)

input_image = torch.rand(3, 28, 28)  # 3 grayscale images
logits = seq_modules(input_image)
print(logits)

tensor([[ 0.2793,  0.2811,  0.0287, -0.1657,  0.0431, -0.0273, -0.0740, -0.2082,
         -0.0024, -0.1331],
        [ 0.1714,  0.2061, -0.1255, -0.2083, -0.0329, -0.0047, -0.0448, -0.2171,
         -0.0115, -0.1937],
        [ 0.3310,  0.3702,  0.0751, -0.1800,  0.0792, -0.0684, -0.0826, -0.1736,
         -0.0031, -0.0932]], grad_fn=<AddmmBackward0>)


In [12]:
# nn.Softmax
softmax = nn.Softmax(dim=1)
pred_prob = softmax(logits)
print(pred_prob)

tensor([[0.1135, 0.0893, 0.0673, 0.1149, 0.1093, 0.0856, 0.1264, 0.0895, 0.0982,
         0.1059],
        [0.1186, 0.0851, 0.0776, 0.1124, 0.1096, 0.0795, 0.1277, 0.0892, 0.0999,
         0.1005],
        [0.1074, 0.1038, 0.0752, 0.0927, 0.1262, 0.0779, 0.1264, 0.0883, 0.1122,
         0.0898]], grad_fn=<SoftmaxBackward0>)


In [ ]:
# Model Parameters(nn.Moduleで定義される)
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")

Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0076,  0.0351, -0.0133,  ..., -0.0354,  0.0249, -0.0166],
        [-0.0311, -0.0184, -0.0338,  ..., -0.0098, -0.0068, -0.0307]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([ 0.0189, -0.0238], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0233,  0.0068,  0.0251,  ..., -0.0120,  0.0262,  0.0002],
        [ 0.0427,  0.0236, -0.0048,  ..., -0.0030,  0.0437,  0.0411]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Size: torch.Size([512]) | Values: tensor([-0.0284,  0.0115], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.4.weight | Size: torch.Size([10, 512]) | Values: tensor([[ 0.0433, -0.0231,  0.0069,  ...,  0.0065, -0.0255,  0.0285],
        [ 0.0099, -0.0292, -0.0262,  ..., -0

# Additioanl Problems from Gemini
- nn.Moduleをsuper().__init__で初期化しているか？
- nn.Flattenは(B, C, H, W) => (B, C * H * W)であることを理解したか
- tensorとmodelを同一デバイスに置いたか

In [20]:
# B, C, W, H = 100, 1, 28, 28のデータを2層のNNでforward処理
class MyNLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(1*28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [21]:
mynlp = MyNLP().to(device)
print(mynlp)

MyNLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [23]:
mnist = torch.rand(100, 1, 28, 28).to(device)
logits = mynlp(mnist)
print(logits.shape)

torch.Size([100, 10])
